In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

In [3]:
# =========================
# CELL 3 — SET PATH DATASET
# =========================
BASE_DIR = "/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset"
CSV_PATH = os.path.join(BASE_DIR, "DR_grading.csv")
IMG_DIR = os.path.join(BASE_DIR, "DR_grading")
OUTPUT_NPZ = os.path.join(BASE_DIR, "DDR_dataset_224.npz")

print("BASE_DIR  :", BASE_DIR)
print("CSV_PATH  :", CSV_PATH)
print("IMG_DIR   :", IMG_DIR)
print("OUTPUT_NPZ:", OUTPUT_NPZ)

BASE_DIR  : /content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset
CSV_PATH  : /content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading.csv
IMG_DIR   : /content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading
OUTPUT_NPZ: /content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DDR_dataset_224.npz


In [4]:
# =========================
# CELL 4 — CHECK PATH
# =========================
print("BASE_DIR exists :", os.path.exists(BASE_DIR))
print("CSV_PATH exists :", os.path.exists(CSV_PATH))
print("IMG_DIR exists  :", os.path.exists(IMG_DIR))

if os.path.exists(IMG_DIR):
    print("\nContoh isi folder image:")
    print(os.listdir(IMG_DIR)[:10])

BASE_DIR exists : True
CSV_PATH exists : True
IMG_DIR exists  : True

Contoh isi folder image:
['20170503083257799.jpg', '20170502092649506.jpg', '20170502160732595.jpg', '20170502002324525.jpg', '20170502103235986.jpg', '20170503160817592.jpg', '20170503090730660.jpg', '20170503172654258.jpg', '20170502112049508.jpg', '20170502091525413.jpg']


In [5]:
# =========================
# CELL 5 — LOAD CSV
# =========================
df = pd.read_csv(CSV_PATH)

print("Shape   :", df.shape)
print("Columns :", df.columns.tolist())
print("\nHead:")
display(df.head())

print("\nDistribusi label:")
print(df["diagnosis"].value_counts().sort_index())

Shape   : (12522, 2)
Columns : ['id_code', 'diagnosis']

Head:


,id_code,diagnosis
0,20170413102628830.jpg,0
1,20170413111955404.jpg,0
2,20170413112015395.jpg,0
3,20170413112017305.jpg,0
4,20170413112528859.jpg,0



Distribusi label:
diagnosis
0    6266
1     630
2    4477
3     236
4     913
Name: count, dtype: int64


In [6]:
# =========================
# CELL 6 — CHECK SAMPLE FILE
# =========================
sample_id = str(df.iloc[0]["id_code"]).strip()

print("Sample ID:", sample_id)

# cek langsung
direct_path = os.path.join(IMG_DIR, sample_id)
print("Direct check:", direct_path, os.path.exists(direct_path))

# cek kalau ditambah ekstensi
possible_exts = [".jpg", ".JPG", ".jpeg", ".JPEG", ".png", ".PNG"]

for ext in possible_exts:
    test_path = os.path.join(IMG_DIR, sample_id + ext)
    print((ext, os.path.exists(test_path), test_path))

Sample ID: 20170413102628830.jpg
Direct check: /content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading/20170413102628830.jpg True
('.jpg', False, '/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading/20170413102628830.jpg.jpg')
('.JPG', False, '/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading/20170413102628830.jpg.JPG')
('.jpeg', False, '/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading/20170413102628830.jpg.jpeg')
('.JPEG', False, '/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading/20170413102628830.jpg.JPEG')
('.png', False, '/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading/20170413102628830.jpg.png')
('.PNG', False, '/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading/20170413102628830.jpg.PNG')


In [7]:
# =========================
# CELL 7 — CONFIG
# =========================
IMG_SIZE = (224, 224)
NORMALIZE = True
SAVE_FILENAMES = True
POSSIBLE_EXTS = [".jpg", ".JPG", ".jpeg", ".JPEG", ".png", ".PNG"]

print("IMG_SIZE       :", IMG_SIZE)
print("NORMALIZE      :", NORMALIZE)
print("SAVE_FILENAMES :", SAVE_FILENAMES)
print("POSSIBLE_EXTS  :", POSSIBLE_EXTS)

IMG_SIZE       : (224, 224)
NORMALIZE      : True
SAVE_FILENAMES : True
POSSIBLE_EXTS  : ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']


In [8]:
# =========================
# CELL 8 — FIND IMAGE FILE
# =========================
def find_image_path(img_dir, img_id, possible_exts):
    img_id = str(img_id).strip()

    # kalau img_id sudah punya ekstensi
    direct_path = os.path.join(img_dir, img_id)
    if os.path.exists(direct_path):
        return direct_path

    # kalau belum, coba tambahkan ekstensi
    for ext in possible_exts:
        candidate = os.path.join(img_dir, img_id + ext)
        if os.path.exists(candidate):
            return candidate

    return None

In [9]:
# =========================
# CELL 9 — LOAD & PREPROCESS IMAGE
# =========================
def load_and_preprocess_image(img_path, target_size=(224, 224), normalize=True):
    img = Image.open(img_path).convert("RGB")
    img = img.resize(target_size)
    img = np.array(img, dtype=np.float32)

    if normalize:
        img = img / 255.0

    return img

In [ ]:
# =========================
# CELL 10 — VALIDATE IMAGE MATCHING
# =========================
missing_files = []
found_count = 0

for img_id in tqdm(df["id_code"], desc="Checking image files"):
    img_path = find_image_path(IMG_DIR, img_id, POSSIBLE_EXTS)
    if img_path is None:
        missing_files.append(str(img_id).strip())
    else:
        found_count += 1

print("Jumlah data CSV     :", len(df))
print("Jumlah file ketemu  :", found_count)
print("Jumlah file missing :", len(missing_files))

if len(missing_files) > 0:
    print("\nContoh missing:")
    print(missing_files[:10])

Checking image files: 100%|██████████| 12522/12522 [00:04<00:00, 2836.63it/s]

Jumlah data CSV     : 12522
Jumlah file ketemu  : 12522
Jumlah file missing : 0


: 

In [11]:
# =========================
# CELL 11 — CONVERT IMAGES TO ARRAY
# =========================
images = []
labels = []
filenames = []
failed_files = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing DDR images"):
    img_id = str(row["id_code"]).strip()
    label = int(row["diagnosis"])

    img_path = find_image_path(IMG_DIR, img_id, POSSIBLE_EXTS)

    if img_path is None:
        failed_files.append((img_id, "file not found"))
        continue

    try:
        img = load_and_preprocess_image(
            img_path=img_path,
            target_size=IMG_SIZE,
            normalize=NORMALIZE
        )

        images.append(img)
        labels.append(label)
        filenames.append(img_id)

    except Exception as e:
        failed_files.append((img_id, str(e)))

X = np.array(images, dtype=np.float32)
y = np.array(labels, dtype=np.int64)
filenames = np.array(filenames)

print("X shape :", X.shape)
print("y shape :", y.shape)
print("failed  :", len(failed_files))

Processing DDR images: 100%|██████████| 12522/12522 [1:13:55<00:00,  2.82it/s]


: 

: 

In [ ]:
# =========================
# CELL 12 — INSPECT RESULT
# =========================
print("Jumlah image berhasil:", len(X))
print("Jumlah label berhasil:", len(y))
print("Jumlah failed        :", len(failed_files))

if len(failed_files) > 0:
    print("\nContoh file gagal:")
    for item in failed_files[:10]:
        print(item)

if len(X) > 0:
    print("\nX dtype   :", X.dtype)
    print("y dtype   :", y.dtype)
    print("Label unik:", np.unique(y))
    print("Min pixel :", X.min())
    print("Max pixel :", X.max())
else:
    print("\nX kosong. Berarti masih ada masalah di path / nama file / ekstensi.")

In [ ]:
# =========================
# CELL 13 — SAVE TO NPZ
# =========================
if len(X) == 0:
    raise ValueError("X kosong, NPZ tidak bisa disimpan. Cek hasil Cell 12.")

if SAVE_FILENAMES:
    np.savez_compressed(
        OUTPUT_NPZ,
        X=X,
        y=y,
        filenames=filenames
    )
else:
    np.savez_compressed(
        OUTPUT_NPZ,
        X=X,
        y=y
    )

print("NPZ berhasil disimpan ke:")
print(OUTPUT_NPZ)

In [ ]:
# =========================
# CELL 14 — TEST LOAD NPZ
# =========================
data = np.load(OUTPUT_NPZ, allow_pickle=True)

print("Keys:", data.files)
print("X shape:", data["X"].shape)
print("y shape:", data["y"].shape)

if "filenames" in data.files:
    print("filenames shape:", data["filenames"].shape)
    print("Contoh filenames:", data["filenames"][:5])

In [ ]:
# =========================
# CELL 15 — SHOW SAMPLE
# =========================
import matplotlib.pyplot as plt

idx = 0
plt.figure(figsize=(4, 4))
plt.imshow(X[idx])
plt.title(f"Label: {y[idx]}, Filename: {filenames[idx]}")
plt.axis("off")
plt.show()